# 导入 (import)

In [9]:
import pandas as pd

"""
处理 jsonl 为 pandas dataframe
Process jsonl to pandas dataframe
"""
words_df = pd.read_json("./crawler_output.jl", orient="records", lines=True)
words_df.head()

,word,part_of_speech,phonetic_symbol,definitions
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '[ C ] US', 'definition': {'en': 'a ..."
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '', 'definition': {'en': 'abbreviati..."
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]","[{'cefr': '', 'definition': {'en': 'the standa..."
3,rivulet,noun [ C ],"[/ˈrɪv.jə.lət/, /ˈrɪv.jə.lət/]","[{'cefr': '', 'definition': {'en': 'a very sma..."
4,riviera,noun [ C ],"[/rɪv.iˈeə.rə/, /rɪv.iˈer.ə/]","[{'cefr': '', 'definition': {'en': 'an area of..."


# 预处理

In [10]:
words_df.rename(columns={"word": "spelling"}, inplace=True)
words_df = words_df.astype(
    {
        "spelling": "string",
        "part_of_speech": "string",
        "phonetic_symbol": object,
        "definitions": object,
    }
)

print(words_df.shape)
words_df.head()

(73220, 4)


,spelling,part_of_speech,phonetic_symbol,definitions
0,RN,noun,"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '[ C ] US', 'definition': {'en': 'a ..."
1,R.N.,noun [ C ],"[/ˌɑːrˈen/, /ˌɑːrˈen/]","[{'cefr': '', 'definition': {'en': 'abbreviati..."
2,riyal,noun [ C ],"[/riˈɑːl/, /riːˈjɑːl/]","[{'cefr': '', 'definition': {'en': 'the standa..."
3,rivulet,noun [ C ],"[/ˈrɪv.jə.lət/, /ˈrɪv.jə.lət/]","[{'cefr': '', 'definition': {'en': 'a very sma..."
4,riviera,noun [ C ],"[/rɪv.iˈeə.rə/, /rɪv.iˈer.ə/]","[{'cefr': '', 'definition': {'en': 'an area of..."


# 展开音标

In [11]:
phonetic_symbols = pd.DataFrame(
    words_df["phonetic_symbol"].tolist(),
    columns=["phonetic_symbol_bre", "phonetic_symbol_ame"],
)


words_df = pd.concat(
    [words_df.drop("phonetic_symbol", axis=1), phonetic_symbols], axis=1
)
words_df = words_df.astype(
    {"phonetic_symbol_bre": "string", "phonetic_symbol_ame": "string"}
)

print(words_df.shape)
words_df.head()

(73220, 5)


,spelling,part_of_speech,definitions,phonetic_symbol_bre,phonetic_symbol_ame
0,RN,noun,"[{'cefr': '[ C ] US', 'definition': {'en': 'a ...",/ˌɑːrˈen/,/ˌɑːrˈen/
1,R.N.,noun [ C ],"[{'cefr': '', 'definition': {'en': 'abbreviati...",/ˌɑːrˈen/,/ˌɑːrˈen/
2,riyal,noun [ C ],"[{'cefr': '', 'definition': {'en': 'the standa...",/riˈɑːl/,/riːˈjɑːl/
3,rivulet,noun [ C ],"[{'cefr': '', 'definition': {'en': 'a very sma...",/ˈrɪv.jə.lət/,/ˈrɪv.jə.lət/
4,riviera,noun [ C ],"[{'cefr': '', 'definition': {'en': 'an area of...",/rɪv.iˈeə.rə/,/rɪv.iˈer.ə/


# 展开释义

In [12]:
assert (
    words_df["definitions"].isnull().sum() == 0
), "数据处理错误，存在缺失的 definitions！Data processing error, missing definitions!"

words_df = words_df.explode("definitions")
definitions = pd.json_normalize(words_df["definitions"])
definitions = definitions.rename(
    columns={"definition.en": "definition_en", "definition.zh": "definition_zh"}
)
definitions = definitions.drop(columns=["cefr"])

words_df = pd.concat([words_df.drop(columns=["definitions"]), definitions], axis=1)
words_df = words_df.astype({"definition_en": "string", "definition_zh": "string"})

assert len(definitions) == len(
    words_df
), "数据处理错误，行数不匹配！Data processing error, row count mismatch!"

print(words_df.shape)
words_df.head()

(99449, 7)


,spelling,part_of_speech,phonetic_symbol_bre,phonetic_symbol_ame,examples,definition_en,definition_zh
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,[{'en': 'An RN makes about $30 an hour in this...,a \n \n registered nurse ; als...,注册护士（可用于姓名后）
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,"[{'en': 'Captain H. Doughty, RN', 'zh': '英国皇家海...",abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）
1,R.N.,noun [ C ],/ˌɑːrˈen/,/ˌɑːrˈen/,[],abbreviation for\n \n registere...,注册护士（registered nurse的缩写）
2,riyal,noun [ C ],/riˈɑːl/,/riːˈjɑːl/,[],the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）
3,rivulet,noun [ C ],/ˈrɪv.jə.lət/,/ˈrɪv.jə.lət/,[{'en': 'Rivulets of sweat/rain/blood ran down...,a very small stream or flow of liquid,小溪;小河;细流


# 展开例子

In [13]:
words_df = words_df.explode("examples")

# 处理 examples 列，空的例子用 {} 填充，方便继续 json normalize 展开
words_df["examples"] = words_df["examples"].apply(
    lambda x: x if isinstance(x, dict) else {}
)
examples = pd.json_normalize(words_df["examples"])
examples = examples.rename(columns={"en": "example_en", "zh": "example_zh"})

words_df = pd.concat([words_df.drop(columns=["examples"]), examples], axis=1)
words_df = words_df.astype({"example_en": "string", "example_zh": "string"})
words_df["example_en"] = words_df["example_en"].fillna("")
words_df["example_zh"] = words_df["example_zh"].fillna("")

assert len(examples) == len(
    words_df
), "数据处理错误，行数不匹配！Data processing error, row count mismatch!"
print(words_df.shape)
words_df.head()

(155003, 8)


,spelling,part_of_speech,phonetic_symbol_bre,phonetic_symbol_ame,definition_en,definition_zh,example_en,example_zh
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a \n \n registered nurse ; als...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a \n \n registered nurse ; als...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂
1,R.N.,noun [ C ],/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for\n \n registere...,注册护士（registered nurse的缩写）,,
2,riyal,noun [ C ],/riˈɑːl/,/riːˈjɑːl/,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,


# 清洗（Cleaning）

In [14]:
import re

def clean(en_definition: str) -> str:
    # 去除多余的行
    # Remove unnecessary new lines
    en_definition = re.sub(r"\s*\n", " ", en_definition)
    
    # 去除没必要的空格
    # Remove unnecessary spaces
    result: str = ""
    str_list = re.split(r"\s+", en_definition)
    str_list_len = len(str_list)
    for i, sub_str in enumerate(str_list):
        if i == str_list_len - 1:
            result += str_list[-1]
        else:
            result += sub_str + " "
    return result

def assert_not_blank_just_empty(s: pd.Series) -> None:
    """
    应该清洗后字符串不是空白，应该是空字符串
    """
    assert s.str.match(r"^\s+$").sum() == 0, "数据处理错误，存在空白的字符串！Data processing error, blank string exists!"


words_df = words_df.reset_index(drop=True)
words_df["spelling"] = words_df["spelling"].apply(clean)
words_df["phonetic_symbol_bre"] = words_df["phonetic_symbol_bre"].apply(clean)
words_df["phonetic_symbol_ame"] = words_df["phonetic_symbol_ame"].apply(clean)
words_df["part_of_speech"] = words_df["part_of_speech"].apply(clean)
words_df["definition_en"] = words_df["definition_en"].apply(clean)
words_df["definition_zh"] = words_df["definition_zh"].apply(clean)
words_df["example_en"] = words_df["example_en"].apply(clean)
words_df["example_zh"] = words_df["example_zh"].apply(clean)
words_df = words_df.drop_duplicates()
# spelling 不可能是空或者者全是空白的字符串
assert_not_blank_just_empty(words_df["phonetic_symbol_bre"])
assert_not_blank_just_empty(words_df["phonetic_symbol_ame"])
assert_not_blank_just_empty(words_df["part_of_speech"])
assert_not_blank_just_empty(words_df["definition_en"])
assert_not_blank_just_empty(words_df["definition_zh"])
assert_not_blank_just_empty(words_df["example_en"])
assert_not_blank_just_empty(words_df["example_zh"])

print(words_df.shape)
words_df.head()

(154170, 8)


,spelling,part_of_speech,phonetic_symbol_bre,phonetic_symbol_ame,definition_en,definition_zh,example_en,example_zh
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。
1,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第
2,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂
3,R.N.,noun [ C ],/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for registered nurse,注册护士（registered nurse的缩写）,,
4,riyal,noun [ C ],/riˈɑːl/,/riːˈjɑːl/,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,


# 拆表为符合数据库定义的 schema 进行导出

In [15]:
import uuid

NAMESPACE_DIC = uuid.uuid5(uuid.NAMESPACE_DNS, "io.github.zadenyip.enlangmemo")


def assert_words_df_not_duplicated() -> bool:
    assert (
        words_df.duplicated().sum() == 0
    ), "数据处理错误，存在重复的行！Data processing error, duplicated rows exist!"


def is_canonical_text(s: str) -> bool:
    """
    判断字符串是否已经符合规范：去首尾空白、内部单空格。
    Check whether text is already canonical: stripped, lowercase, single spaces.
    """
    return s == " ".join(s.strip().split())

def uuid_v5(*parts: str) -> str:
    """
    生成 UUID v5。
    Generate UUID v5.
    如果穿
    """
    for i, part in enumerate(parts):
        if not isinstance(part, str):
            raise TypeError(f"parts[{i}] must be str, got {type(part).__name__}")
        if not is_canonical_text(part):
            raise ValueError(
                f"parts[{i}] is not canonical: {part!r}. "
                "应该本来就清洗完成了，不应该再有不规范的文本了！"
                "It should have been cleaned already, there shouldn't be non-canonical text!"
            )

    name = "|".join(parts)
    return str(uuid.uuid5(NAMESPACE_DIC, name))

def str_is_empty(s: pd.Series) -> pd.Series:
    return s.isnull() | (s == "")

## words 表

In [16]:
ex_words_tb = words_df[["spelling", "phonetic_symbol_bre", "phonetic_symbol_ame"]]

ex_words_tb = ex_words_tb.drop_duplicates(subset=["spelling"]).reset_index(drop=True)

ex_words_tb.insert(
    loc=0, column="word_id", value=ex_words_tb["spelling"].apply(lambda x: uuid_v5(x))
)

ex_words_tb.to_csv(
    "./ex_words_tb.csv", index=False, encoding="utf-8", lineterminator="\n"
)

print(ex_words_tb.shape)
ex_words_tb.head()

(67984, 4)


,word_id,spelling,phonetic_symbol_bre,phonetic_symbol_ame
0,94f4832d-1e60-58fa-9ada-8141b7d9d949,RN,/ˌɑːrˈen/,/ˌɑːrˈen/
1,659d2776-3b1d-58e9-8920-9342bb4d3aec,R.N.,/ˌɑːrˈen/,/ˌɑːrˈen/
2,9eac7ccf-1dc6-5a78-8712-9c7b8702fa0c,riyal,/riˈɑːl/,/riːˈjɑːl/
3,79013a92-90e7-5b90-9002-45a5672d375b,rivulet,/ˈrɪv.jə.lət/,/ˈrɪv.jə.lət/
4,4f0f59de-fd9c-57d5-9912-eca2e39f06c9,riviera,/rɪv.iˈeə.rə/,/rɪv.iˈer.ə/


In [17]:
# 回填
words_df = words_df.merge(
    ex_words_tb[["spelling", "word_id"]],
    on="spelling",
    how="left",
    validate="many_to_one",
)

assert_not_blank_just_empty(words_df["word_id"])
assert_words_df_not_duplicated()

print(words_df.shape)
words_df.head()

(154170, 9)


,spelling,part_of_speech,phonetic_symbol_bre,phonetic_symbol_ame,definition_en,definition_zh,example_en,example_zh,word_id
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。,94f4832d-1e60-58fa-9ada-8141b7d9d949
1,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第,94f4832d-1e60-58fa-9ada-8141b7d9d949
2,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂,94f4832d-1e60-58fa-9ada-8141b7d9d949
3,R.N.,noun [ C ],/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for registered nurse,注册护士（registered nurse的缩写）,,,659d2776-3b1d-58e9-8920-9342bb4d3aec
4,riyal,noun [ C ],/riˈɑːl/,/riːˈjɑːl/,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,,9eac7ccf-1dc6-5a78-8712-9c7b8702fa0c


## words 词性表

In [18]:
ex_words_poses_tb = words_df[["word_id", "part_of_speech"]]
ex_words_poses_tb = ex_words_poses_tb.drop_duplicates(subset=["word_id", "part_of_speech"])

ex_words_poses_tb.insert(
    loc=0,
    column="pose_id",
    value=ex_words_poses_tb.apply(
        lambda row: uuid_v5(row["word_id"], row["part_of_speech"]), axis=1
    ),
)

ex_words_poses_tb.to_csv(
    "./ex_words_poses_tb.csv", index=False, encoding="utf-8", lineterminator="\n"
)
print(ex_words_poses_tb.shape)
ex_words_poses_tb.head()

(73208, 3)


,pose_id,word_id,part_of_speech
0,862ae064-ee6c-5022-b2c0-339e6ffad46c,94f4832d-1e60-58fa-9ada-8141b7d9d949,noun
3,8b1405a4-4e81-5fd0-8dd4-dec26d42a7ea,659d2776-3b1d-58e9-8920-9342bb4d3aec,noun [ C ]
4,1917322a-f399-54ab-a527-e3fc9f409ebb,9eac7ccf-1dc6-5a78-8712-9c7b8702fa0c,noun [ C ]
5,76cf3a48-898c-5dc1-b5db-76624b69e580,79013a92-90e7-5b90-9002-45a5672d375b,noun [ C ]
6,3884a579-fecd-5ef5-ae46-d02a4fda76a9,4f0f59de-fd9c-57d5-9912-eca2e39f06c9,noun [ C ]


In [19]:
# 回填
words_df = words_df.merge(
    ex_words_poses_tb,
    on=["word_id", "part_of_speech"],
    how="left",
    validate="many_to_one",
)
assert_not_blank_just_empty(words_df["pose_id"])
assert_words_df_not_duplicated()

print(words_df.shape)
words_df.head()

(154170, 10)


,spelling,part_of_speech,phonetic_symbol_bre,phonetic_symbol_ame,definition_en,definition_zh,example_en,example_zh,word_id,pose_id
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。,94f4832d-1e60-58fa-9ada-8141b7d9d949,862ae064-ee6c-5022-b2c0-339e6ffad46c
1,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第,94f4832d-1e60-58fa-9ada-8141b7d9d949,862ae064-ee6c-5022-b2c0-339e6ffad46c
2,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂,94f4832d-1e60-58fa-9ada-8141b7d9d949,862ae064-ee6c-5022-b2c0-339e6ffad46c
3,R.N.,noun [ C ],/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for registered nurse,注册护士（registered nurse的缩写）,,,659d2776-3b1d-58e9-8920-9342bb4d3aec,8b1405a4-4e81-5fd0-8dd4-dec26d42a7ea
4,riyal,noun [ C ],/riˈɑːl/,/riːˈjɑːl/,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,,9eac7ccf-1dc6-5a78-8712-9c7b8702fa0c,1917322a-f399-54ab-a527-e3fc9f409ebb


## 释义表

In [20]:
ex_defs_tb = words_df[["pose_id", "definition_en", "definition_zh"]].drop_duplicates()

ex_defs_tb.insert(
    loc=0,
    column="def_id",
    value=ex_defs_tb.apply(
        lambda row: uuid_v5(row["pose_id"], row["definition_en"], row["definition_zh"]),
        axis=1,
    ),
)

ex_defs_tb.to_csv(
    "./ex_defs_tb.csv", index=False, encoding="utf-8", lineterminator="\n"
)

print(ex_defs_tb.shape)
ex_defs_tb.head()

(99244, 4)


,def_id,pose_id,definition_en,definition_zh
0,18d61c1c-5c6e-5a08-9b3f-3f8d3dad7cbd,862ae064-ee6c-5022-b2c0-339e6ffad46c,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）
2,41a54c31-8d68-5d2a-bf04-ac048a12f5a7,862ae064-ee6c-5022-b2c0-339e6ffad46c,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）
3,8b3bd8f3-b4d0-5211-8c38-57972e23c4e5,8b1405a4-4e81-5fd0-8dd4-dec26d42a7ea,abbreviation for registered nurse,注册护士（registered nurse的缩写）
4,569e2cf2-81af-568d-be46-38823a65f64b,1917322a-f399-54ab-a527-e3fc9f409ebb,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）
5,d5dd933b-ac0d-5349-a1b3-feef95c8cb3f,76cf3a48-898c-5dc1-b5db-76624b69e580,a very small stream or flow of liquid,小溪;小河;细流


In [21]:
# 回填
words_df = words_df.merge(
    ex_defs_tb,
    on=["pose_id", "definition_en", "definition_zh"],
    how="left",
    validate="many_to_one",
)

assert_not_blank_just_empty(words_df["def_id"])
assert_words_df_not_duplicated()

print(words_df.shape)
words_df.head()

(154170, 11)


,spelling,part_of_speech,phonetic_symbol_bre,phonetic_symbol_ame,definition_en,definition_zh,example_en,example_zh,word_id,pose_id,def_id
0,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。,94f4832d-1e60-58fa-9ada-8141b7d9d949,862ae064-ee6c-5022-b2c0-339e6ffad46c,18d61c1c-5c6e-5a08-9b3f-3f8d3dad7cbd
1,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,a registered nurse ; also used after the name ...,注册护士（可用于姓名后）,"Alice Brody, RN",注册护士爱丽丝·布罗第,94f4832d-1e60-58fa-9ada-8141b7d9d949,862ae064-ee6c-5022-b2c0-339e6ffad46c,18d61c1c-5c6e-5a08-9b3f-3f8d3dad7cbd
2,RN,noun,/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for Royal Navy: used especially a...,英国皇家海军（尤用于海军军官姓名后，Royal Navy的缩写）,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂,94f4832d-1e60-58fa-9ada-8141b7d9d949,862ae064-ee6c-5022-b2c0-339e6ffad46c,41a54c31-8d68-5d2a-bf04-ac048a12f5a7
3,R.N.,noun [ C ],/ˌɑːrˈen/,/ˌɑːrˈen/,abbreviation for registered nurse,注册护士（registered nurse的缩写）,,,659d2776-3b1d-58e9-8920-9342bb4d3aec,8b1405a4-4e81-5fd0-8dd4-dec26d42a7ea,8b3bd8f3-b4d0-5211-8c38-57972e23c4e5
4,riyal,noun [ C ],/riˈɑːl/,/riːˈjɑːl/,the standard unit of money used in Saudi Arabi...,里亚尔（沙特阿拉伯和卡塔尔的货币单位）,,,9eac7ccf-1dc6-5a78-8712-9c7b8702fa0c,1917322a-f399-54ab-a527-e3fc9f409ebb,569e2cf2-81af-568d-be46-38823a65f64b


## 例句表

In [22]:
examples_tb = words_df[["def_id", "example_en", "example_zh"]].drop_duplicates()

# drop 掉同时没有中英文例句的行
examples_tb = examples_tb[
    ~(str_is_empty(examples_tb["example_en"]) & str_is_empty(examples_tb["example_zh"]))
]

assert (
    examples_tb[
        (str_is_empty(examples_tb["example_en"]) & str_is_empty(examples_tb["example_zh"]))
    ].shape[0]
    == 0
), "数据处理错误，存在缺失的 example_en 或 example_zh！Data processing error, missing example_en or example_zh!"


examples_tb.insert(
    loc=0,
    column="exp_id",
    value=examples_tb.apply(
        lambda row: uuid_v5(row["def_id"], row["example_en"], row["example_zh"]),
        axis=1,
    ),
)
assert_not_blank_just_empty(examples_tb["def_id"])

examples_tb.to_csv(
    "./examples_tb.csv", index=False, encoding="utf-8", lineterminator="\n"
)

print(examples_tb.shape)
examples_tb.head()

(129186, 4)


,exp_id,def_id,example_en,example_zh
0,298bdba9-55f0-5b94-bda2-d9e048ea03a6,18d61c1c-5c6e-5a08-9b3f-3f8d3dad7cbd,An RN makes about $30 an hour in this city.,在这座城市里一名注册护士的时薪约为30美元。
1,38bbee45-23d9-5191-ac51-d71ced8a1ef7,18d61c1c-5c6e-5a08-9b3f-3f8d3dad7cbd,"Alice Brody, RN",注册护士爱丽丝·布罗第
2,9eac1b83-81eb-5f26-a215-ccb22d0a1c45,41a54c31-8d68-5d2a-bf04-ac048a12f5a7,"Captain H. Doughty, RN",英国皇家海军上校 H.道蒂
5,5443d787-5577-5631-821e-c0c8b4ce5ea2,d5dd933b-ac0d-5349-a1b3-feef95c8cb3f,Rivulets of sweat/rain/blood ran down his face.,汗水／雨水／鲜血从他脸上不住地流下来。
6,1388557f-3fcf-5db2-a086-bde193119b34,49f1073e-4093-54d6-827b-06897d14ba94,the French/Italian/Cornish riviera,法国／意大利／科尼什海滨度假胜地
